# KN1GHT — Puzzle Dataset Builder

Fetches chess puzzles from the Lichess Open Puzzle Database, reconstructs full PGN
context for each puzzle by retrieving the source game, and saves a local JSONL file
that the evaluation notebook can load without hitting any external API.

Run this notebook whenever you want to expand or refresh the puzzle dataset.  
The evaluation notebook reads from `.data/puzzles/pgn_puzzles.jsonl` directly.

---

## What each puzzle record contains

```json
{
  "puzzle_id": "AbCd1",
  "game_id": "xKy2Zp",
  "rating": 1450,
  "themes": ["fork", "middlegame"],
  "pgn_context": "1.e4 e5 2.Nf3 ...", // PGN up to (but not including) the puzzle move
  "fen": "r1bq...", // board state at start of puzzle (for validation)
  "best_move_uci": "e2e4", // first solution move in UCI
  "best_move_san": "e4" // first solution move in SAN (from FEN position)
}
```

## Licensing

The Lichess puzzle database and all Lichess game data (including PGNs retrieved via the API)
are released under **CC0 1.0 (Public Domain Dedication)**. Derived datasets may be freely
published without attribution, though acknowledging the source is good practice.

Reference: https://database.lichess.org/ — _"Database exports are released under the
Creative Commons CC0 license. Use them for research, commercial purpose, publication,
anything you like."_

## Rate limits

The Lichess game API allows **~100 requests/minute** without authentication.  
This notebook sleeps 0.7 s between requests to stay safely under that limit.  
For 200 puzzles that is ~2.5 minutes of fetch time.

Set `LICHESS_API_TOKEN` in the environment (optional) to raise the limit to ~900 req/min.


## 1. Setup


In [ ]:
import io
import json
import os
import random
import re
import time
import urllib.request
import urllib.parse
from pathlib import Path

import chess
import chess.pgn
import pandas as pd

ROOT = Path("..").resolve()
PUZZLE_DIR = ROOT / ".data" / "puzzles"
PUZZLE_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = PUZZLE_DIR / "pgn_puzzles.jsonl"

print(f"Output: {OUT_FILE}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# How many puzzles to fetch in this run.
# Existing records in OUT_FILE are preserved — this is additive.
N_NEW_PUZZLES = 200

# Puzzle rating band to draw from.
RATING_MIN = 1200
RATING_MAX = 1900

# Only keep puzzles tagged with at least one of these themes.
# Empty list = no theme filter.
REQUIRED_THEMES = ["middlegame"]

# Exclude puzzles with any of these themes (openings/endgames are memorized
# patterns rather than tactical generalization).
EXCLUDED_THEMES = ["opening", "endgame"]

# Seconds to sleep between Lichess game API calls.
SLEEP_BETWEEN_REQUESTS = 0.7

# Optional: set LICHESS_API_TOKEN in env for higher rate limits.
LICHESS_TOKEN = os.environ.get("LICHESS_API_TOKEN", "")

# HuggingFace dataset repo to publish to (set to None to skip publishing).
# Format: "<your-hf-username>/chess-puzzles-pgn"
HF_REPO_ID = None  # e.g. "InterwebAlchemy/chess-puzzles-pgn"

print(f"Target: {N_NEW_PUZZLES} new puzzles")
print(f"Rating: {RATING_MIN}–{RATING_MAX}")
print(f"Required themes: {REQUIRED_THEMES or 'any'}")
print(f"Excluded themes: {EXCLUDED_THEMES}")
print(f"Auth:   {'token set' if LICHESS_TOKEN else 'unauthenticated (100 req/min)'}")
print(f"HF repo: {HF_REPO_ID or '(not configured — skip publishing)'}")

## 2. Load or Download Lichess Puzzle CSV

The Lichess puzzle database is a ~150MB CSV (compressed ~30MB).  
It is downloaded once to `.data/puzzles/lichess_puzzles.csv.zst` and re-used on subsequent runs.

Columns: `PuzzleId, FEN, Moves, Rating, RatingDeviation, Popularity, NbPlays, Themes, GameUrl, OpeningTags`


In [ ]:
PUZZLE_CSV_ZST = PUZZLE_DIR / "lichess_puzzles.csv.zst"
PUZZLE_CSV_URL = "https://database.lichess.org/lichess_db_puzzle.csv.zst"


def download_with_progress(url: str, dest: Path) -> None:
    """Download url → dest, printing MB progress."""
    print(f"Downloading {url}")
    req = urllib.request.Request(
        url, headers={"User-Agent": "kn1ght-puzzle-builder/1.0"}
    )
    with urllib.request.urlopen(req) as resp:
        total = int(resp.headers.get("Content-Length", 0))
        downloaded = 0
        chunk_size = 1 << 20  # 1 MB
        with open(dest, "wb") as f:
            while True:
                chunk = resp.read(chunk_size)
                if not chunk:
                    break
                f.write(chunk)
                downloaded += len(chunk)
                pct = downloaded / total * 100 if total else 0
                print(
                    f"  {downloaded / 1e6:.1f} MB / {total / 1e6:.1f} MB ({pct:.0f}%)   ",
                    end="\r",
                )
    print(f"\nSaved to {dest}")


if not PUZZLE_CSV_ZST.exists():
    download_with_progress(PUZZLE_CSV_URL, PUZZLE_CSV_ZST)
else:
    size_mb = PUZZLE_CSV_ZST.stat().st_size / 1e6
    print(f"Using cached puzzle CSV: {PUZZLE_CSV_ZST} ({size_mb:.1f} MB)")

In [ ]:
# zstandard is required to decompress the Lichess CSV.
# Install with: uv add zstandard
try:
    import zstandard as zstd
except ImportError:
    raise ImportError("uv add zstandard")

print("Loading puzzle CSV...")
dctx = zstd.ZstdDecompressor()
with open(PUZZLE_CSV_ZST, "rb") as fh:
    with dctx.stream_reader(fh) as reader:
        puzzles_df = pd.read_csv(
            reader,
            names=[
                "PuzzleId",
                "FEN",
                "Moves",
                "Rating",
                "RatingDeviation",
                "Popularity",
                "NbPlays",
                "Themes",
                "GameUrl",
                "OpeningTags",
            ],
            header=0,
        )
print(f"Total puzzles: {len(puzzles_df):,}")
puzzles_df.head(2)

## 3. Filter and Sample


In [ ]:
def has_required_theme(themes_str: str) -> bool:
    if not REQUIRED_THEMES:
        return True
    themes = set(str(themes_str).split())
    return bool(themes & set(REQUIRED_THEMES))


def has_excluded_theme(themes_str: str) -> bool:
    themes = set(str(themes_str).split())
    return bool(themes & set(EXCLUDED_THEMES))


# ── Rating filter ──────────────────────────────────────────────────────────────
mask = (puzzles_df["Rating"] >= RATING_MIN) & (puzzles_df["Rating"] <= RATING_MAX)
filtered = puzzles_df[mask].copy()
print(f"After rating filter ({RATING_MIN}–{RATING_MAX}): {len(filtered):,}")

# ── Theme filters ──────────────────────────────────────────────────────────────
if REQUIRED_THEMES:
    filtered = filtered[filtered["Themes"].apply(has_required_theme)]
    print(f"After required-theme filter: {len(filtered):,}")

filtered = filtered[~filtered["Themes"].apply(has_excluded_theme)]
print(f"After excluded-theme filter:  {len(filtered):,}")

# ── Exclude puzzle IDs already in OUT_FILE ─────────────────────────────────────
existing_ids: set[str] = set()
if OUT_FILE.exists():
    with open(OUT_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                existing_ids.add(json.loads(line)["puzzle_id"])
    print(f"Skipping {len(existing_ids)} already-fetched puzzles")
    filtered = filtered[~filtered["PuzzleId"].isin(existing_ids)]
    print(f"Remaining candidates: {len(filtered):,}")

# ── Sample (no fixed seed — each run draws independently from the remaining pool) ──
n_sample = min(N_NEW_PUZZLES, len(filtered))
sample = filtered.sample(n=n_sample).reset_index(drop=True)
print(f"\nSampled {n_sample} puzzles for this run")

## 4. PGN Reconstruction

For each sampled puzzle:

1. Extract the Lichess game ID from `GameUrl`
2. Fetch the game PGN via the Lichess API
3. Walk the game move by move until the board FEN matches the puzzle FEN
4. The PGN string up to that point is the puzzle context
5. Convert the first solution move from UCI to SAN

> **Note on Lichess puzzle FEN semantics:** the FEN is the position _before_ the puzzle move.  
> `Moves` is space-separated UCI; the **first** entry is the solution move.
> (Unlike some older Lichess puzzle exports that prepended the opponent's blunder move.)


In [ ]:
def extract_game_id(game_url: str) -> str:
    """Extract the 8-char game ID from any Lichess game URL.

    Handles all URL variants:
      https://lichess.org/AbCdEfGh
      https://lichess.org/AbCdEfGh/black
      https://lichess.org/AbCdEfGh/white
      https://lichess.org/AbCdEfGh#55
      https://lichess.org/AbCdEfGh/black#36
    The game ID is always the path segment immediately after lichess.org/.
    """
    # Split on / and take the 4th segment (index 3): https: / / lichess.org / <id>
    return game_url.split("/")[3].split("#")[0]


def fetch_game_pgn(game_id: str, token: str = "") -> str | None:
    """
    Fetch a single Lichess game PGN by game ID.
    Returns None on any network or HTTP error.
    """
    url = f"https://lichess.org/game/export/{game_id}?moves=true&clocks=false&evals=false&opening=false"
    headers = {
        "Accept": "application/x-chess-pgn",
        "User-Agent": "kn1ght-puzzle-builder/1.0",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.read().decode("utf-8")
    except Exception as exc:
        print(f"    fetch error ({game_id}): {exc}")
        return None


def normalise_fen(fen: str) -> str:
    """Strip the halfmove clock and fullmove number so two FENs of the same
    position compare equal regardless of game history."""
    parts = fen.split()
    return " ".join(parts[:4])  # piece placement, side to move, castling, en passant


def reconstruct_pgn_context(pgn_text: str, target_fen: str) -> str | None:
    """
    Walk a PGN game and return the move-text prefix (no headers) up to
    but NOT including the move that reaches `target_fen`.

    Returns None if the FEN is never reached.
    """
    target_norm = normalise_fen(target_fen)
    game = chess.pgn.read_game(io.StringIO(pgn_text))
    if game is None:
        return None

    board = game.board()
    moves_played: list[str] = []

    for node in game.mainline():
        # Check BEFORE pushing the move — puzzle FEN is the position we're about to move from
        if normalise_fen(board.fen()) == target_norm:
            return _moves_to_pgn(moves_played)
        moves_played.append(board.san(node.move))
        board.push(node.move)

    # Also check the final position
    if normalise_fen(board.fen()) == target_norm:
        return _moves_to_pgn(moves_played)

    return None


def _moves_to_pgn(half_moves: list[str]) -> str:
    """Convert a flat SAN list → numbered PGN string (e.g. '1.e4 e5 2.Nf3')."""
    parts = []
    for i, move in enumerate(half_moves):
        if i % 2 == 0:
            parts.append(f"{i // 2 + 1}.{move}")
        else:
            parts.append(move)
    return " ".join(parts)


def uci_to_san(uci_move: str, fen: str) -> str | None:
    """Convert a UCI move string to SAN given the position FEN."""
    try:
        board = chess.Board(fen)
        move = chess.Move.from_uci(uci_move)
        if move not in board.legal_moves:
            return None
        return board.san(move)
    except Exception:
        return None


print("Helpers defined.")

## 5. Fetch and Write


In [ ]:
skipped_no_pgn = 0
skipped_no_match = 0
skipped_bad_move = 0
written = 0

with open(OUT_FILE, "a") as out_fh:
    for idx, row in sample.iterrows():
        puzzle_id = row["PuzzleId"]
        fen = row["FEN"]
        moves_uci = str(row["Moves"]).split()
        game_url = row["GameUrl"]
        rating = int(row["Rating"])
        themes = str(row["Themes"]).split()

        print(f"[{idx + 1}/{n_sample}] {puzzle_id}  rating={rating}  {game_url}")

        # ── Validate we have a solution move ──────────────────────────────────
        if not moves_uci:
            print("    skip: no moves in record")
            skipped_bad_move += 1
            continue

        best_uci = moves_uci[0]
        best_san = uci_to_san(best_uci, fen)
        if best_san is None:
            print(f"    skip: UCI→SAN conversion failed ({best_uci})")
            skipped_bad_move += 1
            continue

        # ── Fetch source game ─────────────────────────────────────────────────
        game_id = extract_game_id(game_url)
        pgn_text = fetch_game_pgn(game_id, token=LICHESS_TOKEN)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

        if pgn_text is None:
            print("    skip: game fetch failed")
            skipped_no_pgn += 1
            continue

        # ── Reconstruct PGN context ───────────────────────────────────────────
        pgn_context = reconstruct_pgn_context(pgn_text, fen)
        if pgn_context is None:
            print(f"    skip: FEN not found in game PGN")
            skipped_no_match += 1
            continue

        # ── Write record ──────────────────────────────────────────────────────
        record = {
            "puzzle_id": puzzle_id,
            "game_id": game_id,
            "rating": rating,
            "themes": themes,
            "pgn_context": pgn_context,
            "fen": fen,
            "best_move_uci": best_uci,
            "best_move_san": best_san,
        }
        out_fh.write(json.dumps(record) + "\n")
        out_fh.flush()
        written += 1
        print(f"    OK: context={len(pgn_context.split())} moves, answer={best_san}")


print()
print(f"{'─' * 50}")
print(f"Written:          {written}")
print(f"Skipped (fetch):  {skipped_no_pgn}")
print(f"Skipped (no FEN): {skipped_no_match}")
print(f"Skipped (move):   {skipped_bad_move}")
print(f"Total in file:    {written + len(existing_ids)}")

## 6. Validate and Inspect

Load the full dataset and do a quick sanity check.


In [ ]:
records = []
with open(OUT_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Total records in dataset: {len(df)}")
print()
print(
    df[["puzzle_id", "rating", "best_move_san"]]
    .assign(pgn_len=df["pgn_context"].str.split().str.len())
    .describe()
)

In [ ]:
# ── Theme distribution ─────────────────────────────────────────────────────────
from collections import Counter

theme_counts = Counter(t for row in df["themes"] for t in row)
print("Theme distribution:")
for theme, count in theme_counts.most_common(20):
    print(f"  {theme:<30} {count}")

In [ ]:
# ── Spot-check 3 random records ────────────────────────────────────────────────
for rec in random.Random(0).sample(records, min(3, len(records))):
    board = chess.Board(rec["fen"])
    move = board.parse_san(rec["best_move_san"])
    legal = move in board.legal_moves

    print(f"Puzzle {rec['puzzle_id']}  rating={rec['rating']}  themes={rec['themes']}")
    print(
        f"  PGN context ({len(rec['pgn_context'].split())} tokens): {rec['pgn_context'][-80:]}"
    )
    print(
        f"  Best move: {rec['best_move_san']} (UCI: {rec['best_move_uci']})  legal={legal}"
    )
    print()

## 7. Export to HuggingFace Dataset

Converts the JSONL file to a HuggingFace `DatasetDict` with train/validation/test splits
and writes it to disk. Optionally pushes to the Hub.

**Schema** (matches what the evaluation notebook expects):

| Column          | Type         | Description                                         |
| --------------- | ------------ | --------------------------------------------------- |
| `puzzle_id`     | string       | Lichess puzzle ID                                   |
| `game_id`       | string       | Lichess game ID (source of PGN)                     |
| `rating`        | int32        | Puzzle difficulty (Lichess Glicko-2)                |
| `themes`        | list[string] | Tactical theme tags                                 |
| `pgn_context`   | string       | PGN move text up to (not including) the puzzle move |
| `fen`           | string       | Board position at start of puzzle (for validation)  |
| `best_move_uci` | string       | Correct first move in UCI notation                  |
| `best_move_san` | string       | Correct first move in SAN notation                  |

**Splits**: 80% train / 10% validation / 10% test, shuffled with seed=42.


In [ ]:
try:
    from datasets import Dataset, DatasetDict, Features, Value, Sequence
except ImportError:
    raise ImportError("uv add datasets")

# ── Define schema explicitly so HF infers correct types ───────────────────────
FEATURES = Features(
    {
        "puzzle_id": Value("string"),
        "game_id": Value("string"),
        "rating": Value("int32"),
        "themes": Sequence(Value("string")),
        "pgn_context": Value("string"),
        "fen": Value("string"),
        "best_move_uci": Value("string"),
        "best_move_san": Value("string"),
    }
)

# ── Load and split ─────────────────────────────────────────────────────────────
full_ds = Dataset.from_list(records, features=FEATURES)
print(f"Full dataset: {len(full_ds)} records")

# 80 / 10 / 10 split
train_testval = full_ds.train_test_split(test_size=0.20, seed=42)
testval = train_testval["test"].train_test_split(test_size=0.50, seed=42)

dataset_dict = DatasetDict(
    {
        "train": train_testval["train"],
        "validation": testval["train"],
        "test": testval["test"],
    }
)

print(dataset_dict)

# ── Save to disk ───────────────────────────────────────────────────────────────
HF_DISK_PATH = PUZZLE_DIR / "hf_dataset"
dataset_dict.save_to_disk(str(HF_DISK_PATH))
print(f"\nSaved to disk: {HF_DISK_PATH}")

### 7a. Generate Dataset Card (README.md)

The dataset card is written to `.data/puzzles/hf_dataset/README.md`.  
When publishing to the Hub, this becomes the dataset page description.


In [ ]:
from collections import Counter as _Counter

# Compute summary stats for the card
_theme_counts = _Counter(t for row in df["themes"] for t in row)
_top_themes = ", ".join(f"`{t}`" for t, _ in _theme_counts.most_common(8))
_n_total = len(df)
_rating_mean = int(df["rating"].mean())
_rating_range = f"{int(df['rating'].min())}–{int(df['rating'].max())}"

DATASET_CARD = f"""---
license: cc0-1.0
language:
  - en
tags:
  - chess
  - pgn
  - puzzles
  - tactics
pretty_name: Chess Puzzles with PGN Context
size_categories:
  - 1K<n<10K
---

# Chess Puzzles with PGN Context

Tactical chess puzzles drawn from the [Lichess Open Puzzle Database](https://database.lichess.org/#puzzles),
augmented with full PGN game context reconstructed from the source Lichess games.

Each record presents a middlegame position as a PGN move sequence — the same format used
to train PGN language models like [KN1GHT](https://github.com/InterwebAlchemy/kn1ght) — together
with the engine-validated best move as the label.

## Dataset Summary

- **{_n_total:,} puzzles** with reconstructed PGN context
- Rating range: {_rating_range} (mean: {_rating_mean})
- Themes: {_top_themes} (middlegame only; opening/endgame excluded)
- Splits: 80% train / 10% validation / 10% test

## Schema

| Column | Type | Description |
|---|---|---|
| `puzzle_id` | string | Lichess puzzle ID |
| `game_id` | string | Lichess game ID (source of PGN context) |
| `rating` | int32 | Puzzle difficulty (Lichess Glicko-2 rating) |
| `themes` | list[string] | Tactical theme tags (e.g. `fork`, `pin`, `skewer`) |
| `pgn_context` | string | PGN move text up to (not including) the puzzle move |
| `fen` | string | Board position at start of puzzle in FEN notation |
| `best_move_uci` | string | Correct first move in UCI notation |
| `best_move_san` | string | Correct first move in SAN notation |

## Usage

```python
from datasets import load_dataset

ds = load_dataset("InterwebAlchemy/chess-puzzles-pgn")

# Each example:
# {{'pgn_context': '1.e4 e5 2.Nf3 Nc6 ... 18.Rxd4',
#   'best_move_san': 'Nf6+',
#   'rating': 1487,
#   'themes': ['fork', 'middlegame']}}
```

## How PGN context is reconstructed

For each Lichess puzzle:
1. The source game PGN is fetched from `lichess.org/api/game/<id>`
2. The game is replayed move by move until the board FEN matches the puzzle FEN
3. The move-text up to that point is stored as `pgn_context`
4. The first solution move (UCI → SAN) is stored as `best_move_san`

Puzzles where the FEN could not be located in the source game are discarded.

## Intended use

Evaluating and fine-tuning PGN language models on tactical positions. The `pgn_context`
field can be fed directly to any model that generates chess moves as PGN continuations.

## Licensing

This dataset is derived from the [Lichess Open Database](https://database.lichess.org/),
which is released under [CC0 1.0 (Public Domain)](https://creativecommons.org/publicdomain/zero/1.0/).
This derived dataset is also released under CC0 1.0.
"""

(HF_DISK_PATH / "README.md").write_text(DATASET_CARD)
print(f"Dataset card written to {HF_DISK_PATH / 'README.md'}")
print()
print(DATASET_CARD[:800], "...")

### 7b. Push to HuggingFace Hub (hold — validate dataset first)

**Do not run until the dataset has been spot-checked and the evaluation notebook
has been run against it at least once.**

When ready: set `HF_REPO_ID` at the top of the notebook and run `huggingface-cli login`.
The dataset card generated in 7a is automatically picked up as the repo README.


In [ ]:
# ── Publishing is intentionally disabled until the dataset is validated ──────────
# To publish:
#   1. Run the evaluation notebook against this dataset (Phase C')
#   2. Spot-check a sample of records manually
#   3. Set HF_REPO_ID = "<your-username>/chess-puzzles-pgn" in the config cell
#   4. Run: huggingface-cli login
#   5. Uncomment and run the block below

# if HF_REPO_ID is None:
#     print("HF_REPO_ID is not set — skipping Hub push.")
# else:
#     print(f"Pushing to Hub: {HF_REPO_ID}")
#     dataset_dict.push_to_hub(
#         HF_REPO_ID,
#         commit_message=f"Add {_n_total} puzzles (rating {_rating_range}, middlegame)",
#     )
#     print(f"\nPublished: https://huggingface.co/datasets/{HF_REPO_ID}")

print("Publishing skipped — validate dataset before uncommenting above.")

## 8. Cache wtharvey FEN Puzzles (Phase C)

The evaluation notebook's Phase C uses FEN puzzles from wtharvey.com — kept as FEN
intentionally to measure the generalization gap (KN1GHT expected ~0%).
This cell pre-fetches and caches them so the evaluation notebook needs no live requests.


In [ ]:
import urllib.request as _req

WTHARVEY_URL = "https://wtharvey.com/m8n2.txt"
WTHARVEY_CACHE = PUZZLE_DIR / "wtharvey_fen_puzzles.json"
N_WTHARVEY = 10  # match the evaluation notebook default
WTHARVEY_SEED = 42


def fetch_and_cache_wtharvey(url: str, cache: Path, n: int, seed: int) -> list[dict]:
    import re as _re

    with _req.urlopen(url, timeout=15) as resp:
        text = resp.read().decode("utf-8")

    puzzles = []
    lines = text.split("\n")
    for i, line in enumerate(lines):
        line = line.strip()
        if line.count("/") == 7 and (" w " in line or " b " in line):
            citation = lines[i - 1].strip() if i > 0 else ""
            fen = line
            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            solution = lines[j].strip() if j < len(lines) else ""
            m = _re.search(r"\d+\.\s+(\S+)", solution)
            first_move = m.group(1) if m else None
            if first_move and citation and fen:
                puzzles.append(
                    {
                        "citation": citation,
                        "fen": fen,
                        "solution": solution,
                        "best_move": first_move,
                    }
                )

    sample = random.Random(seed).sample(puzzles, min(n, len(puzzles)))
    cache.write_text(json.dumps(sample, indent=2))
    print(f"Cached {len(sample)} wtharvey puzzles → {cache}")
    return sample


if WTHARVEY_CACHE.exists():
    print(f"wtharvey cache already exists: {WTHARVEY_CACHE}")
else:
    fetch_and_cache_wtharvey(WTHARVEY_URL, WTHARVEY_CACHE, N_WTHARVEY, WTHARVEY_SEED)

---

## Loading in the evaluation notebook

Replace the `fetch_wtharvey_puzzles(...)` call and the planned Lichess fetch with:

```python
from datasets import load_dataset

# ── PGN puzzles (Phase C') — load from Hub or local disk ──────────────────────
# From Hub (after publishing):
pgn_ds = load_dataset("InterwebAlchemy/chess-puzzles-pgn", split="test")
PGN_PUZZLES = random.Random(42).sample(list(pgn_ds), min(20, len(pgn_ds)))

# From local disk (before publishing / offline):
# from datasets import load_from_disk
# pgn_ds = load_from_disk(".data/puzzles/hf_dataset")["test"]
# PGN_PUZZLES = random.Random(42).sample(list(pgn_ds), min(20, len(pgn_ds)))

# ── FEN puzzles (Phase C) — load from cache ────────────────────────────────────
FEN_PUZZLE_FILE = ROOT / ".data" / "puzzles" / "wtharvey_fen_puzzles.json"
FEN_PUZZLES = json.loads(FEN_PUZZLE_FILE.read_text())
```
